In [1]:
%pylab inline

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [2]:
from pycbc.events import hm_utils

In [4]:
tlen=8192
t2_coinc_window=8
t3_coinc_window=17

In [10]:
from numba import njit

In [24]:
@njit
def get_indices_jit_3_ifo(tlen, t2_coinc_window, t3_coinc_window, dtype=np.int64):
    idx = np.array([
        [i,j,k]
            for i in range(tlen)
                for j in range(max(i-t2_coinc_window, 0), min(tlen, i+t2_coinc_window+1))
                    for k in range(max(i-t3_coinc_window, 0), min(tlen, i+t3_coinc_window+1))
                ], dtype=dtype)
    return idx

@njit
def get_indices_jit_2_ifo(tlen, t2_coinc_window, dtype=np.int64):
    idx = np.array([
        [i,j]
            for i in range(tlen)
                for j in range(max(i-t2_coinc_window, 0), min(tlen, i+t2_coinc_window+1))
                ], dtype=dtype)
    return idx

In [26]:
det_idx = get_indices_2det(tlen, t2_coinc_window)

In [19]:
len(det_idx)

139192

In [62]:
det_idx = hm_utils.index_combinations(
    tlen,
    t2_coinc_window,
    t3_coinc_window)

In [21]:
len(det_idx)

4867574

In [61]:
def index_combinations(tlen, t2_coinc_window, t3_coinc_window, dtype=np.int64):
    """For three detectors, this is equivalent to calling get_indices_jit_3_ifo, but is 
    generally faster. For two detectors this just calls get_indices_jit_2_ifo directly."""
    if t3_coinc_window is None:
        return get_indices_jit_2_ifo(tlen, t2_coinc_window, dtype)
    elif 2*max(t2_coinc_window, t3_coinc_window) > tlen:
        return get_indices_jit_3_ifo(tlen, t2_coinc_window, t3_coinc_window, dtype)
    # forgetting the tails at first
    largest_window = max(t2_coinc_window, t3_coinc_window)
    idx_1_mid = np.arange(tlen - 2*(t3_coinc_window+1), dtype=dtype).repeat( \
        (2*t2_coinc_window+1)*(2*t3_coinc_window+1) \
    ) + largest_window + 1
    idx_2_mid = idx_1_mid + np.tile(
        np.arange(-t2_coinc_window, t2_coinc_window+1, dtype=dtype).repeat(2*t3_coinc_window+1), 
        (tlen - 2*(t3_coinc_window+1))
    )
    idx_3_mid = idx_1_mid + np.tile(
        np.arange(-t3_coinc_window, t3_coinc_window+1, dtype=dtype), 
        (tlen - 2*(t3_coinc_window+1)) * (2*t2_coinc_window+1)
    )
    idx_1_ends, idx_2_ends, idx_3_ends = get_indices_jit(
        2*(t3_coinc_window+1), t2_coinc_window, t3_coinc_window, dtype=dtype).T
    # now get the tails
    n_start = int(len(idx_1_ends) / 2)
    idx_1 = np.concatenate((idx_1_ends[:n_start], idx_1_mid, idx_1_ends[-n_start:] + tlen - (2*t3_coinc_window + 2)))
    idx_2 = np.concatenate((idx_2_ends[:n_start], idx_2_mid, idx_2_ends[-n_start:] + tlen - (2*t3_coinc_window + 2)))
    idx_3 = np.concatenate((idx_3_ends[:n_start], idx_3_mid, idx_3_ends[-n_start:] + tlen - (2*t3_coinc_window + 2)))
    return np.array([idx_1, idx_2, idx_3]).T

In [65]:
det_idx = det_idx[:,0]

In [74]:
np.array([det_idx]).T

array([[   0],
       [   0],
       [   0],
       ...,
       [8191],
       [8191],
       [8191]])

In [77]:
np.random.choice([0,1], size=19)

array([1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0])

### sum and threshold

In [51]:
def get_index_array_dtype(max_index):
    # reduce the size of the index array where possible
    bits_dtypes = [(8, np.uint8), (16, np.uint16), (32, np.uint32), (64, np.uint64)]
    for bits, dtype in bits_dtypes: 
        if 2**bits > max_index+1: break
    return dtype

@njit
def three_det_sum_idx_jit(t1, t2, t3, idx):
    """Most efficient if you are using numba and have the indexes."""
    temp = np.array([
        t1[i] + t2[j] + t3[k]
            for i,j,k in idx
                ])
    return temp

@njit
def two_det_sum_idx_jit(t1, t2, idx):
    return array([t1[i] + t2[j] for i,j in idx])

def detector_sum_and_threshold(snr_2_filt_rss, idx, threshold):
    snr_2_filt_rss = abs(snr_2_filt_rss)**2
    nifos = len(snr_2_filt_rss)
    dtype = get_index_array_dtype(np.max(idx))
    if nifos == 3:
        network_snr_sq = three_det_sum_idx_jit(
            snr_2_filt_rss[0], snr_2_filt_rss[1], snr_2_filt_rss[2], idx.astype(dtype))
    elif nifos == 2:
        network_snr_sq = two_det_sum_idx_jit(
            snr_2_filt_rss[0], snr_2_filt_rss[1], idx.astype(dtype))
    mask = network_snr_sq > threshold**2
    return np.sqrt(network_snr_sq[mask]), idx[mask]

In [57]:
N=5120
t1 = array(uniform(0,1,N), dtype=float32)
t2 = array(uniform(0,1,N), dtype=float32)
t3 = array(uniform(0,0.6,N), dtype=float32)
snr_2_filt_rss = np.array([t1, t2, t3])
snr_2_filt_rss = np.array([t1, t2])

In [59]:
# idx = index_combinations(tlen, t2_coinc_window, t3_coinc_window)
idx = index_combinations(tlen, t2_coinc_window, None)

In [60]:
detector_sum_and_threshold(snr_2_filt_rss, idx, 0.9)

(array([1.1681192, 1.1759907, 1.1171671, ..., 1.1645868, 1.1815144,
        1.1724827], dtype=float32),
 array([[   0,    0],
        [   0,    2],
        [   0,    3],
        ...,
        [8191, 8189],
        [8191, 8190],
        [8191, 8191]]))